# **Lab 5: Build and Train LSTM for Text Classification & Sequence-to-Sequence**
This notebook demonstrates:
1. LSTM for Text Classification (IMDB Dataset)
2. Sequence-to-Sequence (Encoder-Decoder LSTM)


# **Text Classification Architectures**



| Layer / Step      | **RNN**                       | **LSTM**                      | **GRU**                       |
|------------------|-------------------------------|-------------------------------|-------------------------------|
| **Input**         | Text sequence (`max_len`)     | Text sequence (`max_len`)     | Text sequence (`max_len`)     |
| **Tokenization**  | Words → integers              | Words → integers              | Words → integers              |
| **Padding**       | Pad to `max_len`              | Pad to `max_len`              | Pad to `max_len`              |
| **Embedding**     | `Embedding(input_dim, output_dim, input_length)` | `Embedding(input_dim, output_dim, input_length)` | `Embedding(input_dim, output_dim, input_length)` |
| **Recurrent**     | `SimpleRNN(32)`               | `LSTM(64)`                    | `GRU(64)`                     |
| **Dropout**       | `Dropout(0.5)`                | `Dropout(0.5)`                | `Dropout(0.5)`                |
| **Output**        | `Dense(1, activation='sigmoid')` | `Dense(1, activation='sigmoid')` | `Dense(1, activation='sigmoid')` |
| **Loss Function** | Binary cross-entropy           | Binary cross-entropy           | Binary cross-entropy           |
| **Optimizer**     | Adam                          | Adam                          | Adam                          |
| **Metrics**       | Accuracy                       | Accuracy                       | Accuracy                       |

# **Built-in Keras Recurrent Layers**

| Layer        | Built-in Keras Layer                                      | Key Parameters |
|-------------|-----------------------------------------------------------|----------------|
| **SimpleRNN** | `tf.keras.layers.SimpleRNN(units=32, activation='tanh', return_sequences=False, dropout=0.0, recurrent_dropout=0.0)` | units, activation, return_sequences, dropout, recurrent_dropout |
| **LSTM**     | `tf.keras.layers.LSTM(units=64, activation='tanh', recurrent_activation='sigmoid', return_sequences=False, dropout=0.0, recurrent_dropout=0.0)` | units, activation, recurrent_activation, return_sequences, dropout, recurrent_dropout |
| **GRU**      | `tf.keras.layers.GRU(units=64, activation='tanh', recurrent_activation='sigmoid', return_sequences=False, dropout=0.0, recurrent_dropout=0.0)` | units, activation, recurrent_activation, return_sequences, dropout, recurrent_dropout |

.

| Parameter / Layer        | **SimpleRNN**                                                                                  | **LSTM**                                                                                                      | **GRU**                                                                                                      |
|--------------------------|------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------|
| **Layer**                | `tf.keras.layers.SimpleRNN(...)`                                                               | `tf.keras.layers.LSTM(...)`                                                                                  | `tf.keras.layers.GRU(...)`                                                                                  |
| **Units**                | 32                                                                                             | 64                                                                                                          | 64                                                                                                          |
| **Activation**           | `tanh`                                                                                         | `tanh`                                                                                                      | `tanh`                                                                                                      |
| **Recurrent Activation** | None (no gates)                                                                                 | `sigmoid`                                                                                                   | `sigmoid`                                                                                                   |
| **Return Sequences**     | `False`                                                                                        | `False`                                                                                                     | `False`                                                                                                     |
| **Dropout**              | 0.0                                                                                            | 0.0                                                                                                         | 0.0                                                                                                         |
| **Recurrent Dropout**    | 0.0                                                                                            | 0.0                                                                                                         | 0.0                                                                                                         |

# **BASIC RNN(RECURRENT NEURAL NETWORK)**

 **RNN PIPELINE -**

 **`Raw text`** → **`Tokenization & Padding`** → **`Train/Test Split`** → **`Embedding`** → **`SimpleRNN`** → **`Dropout`** → **`Dense`** → **`Train`** → **`Evaluate`** → **`Predict Sentiment`**

**Step 1: Import Libraries**

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from sklearn.model_selection import train_test_split

**Step 2: Sample Data**

In [2]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative (label mixed sentences as positive for simplicity)

**Step 3: Tokenize and Pad Sequences**

In [3]:
max_words = 1000                   # Maximum number of words to keep in vocabulary  --- Only top 1000 most frequent words will be considered
max_len = 15                       # Maximum length of each sequence (sentence) ----  All sequences will be padded/truncated to length 15

# Create tokenizer
# oov_token = "<OOV>" will replace unknown words
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")       # num_words → limit vocab size to 1000        # oov_token → replace unknown words with "<OOV>"

# Fit tokenizer on texts
# Example:
# "students of aimlc are brilliant"
# → {'students':2, 'of':3, 'aimlc':4, 'are':5, 'brilliant':6}
tokenizer.fit_on_texts(texts)                                       # Assigns a unique integer to each word

# Convert texts to sequences
# Example:
# "students of aimlc are brilliant"
# → [2, 3, 4, 5, 6]
sequences = tokenizer.texts_to_sequences(texts)                     # Convert each sentence into sequence of integers

# Pad sequences to ensure same length (15)
# padding='post' → add zeros at the end if sentence is short
# Longer sentences will be truncated
# Apply padding
# max_len = 10
# padding='post' → add zeros at end
# Example:
# [2, 3, 4, 5, 6]
# → [2, 3, 4, 5, 6, 0, 0, 0, 0, 0]
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

# Print example output
# Example output:
# "students of aimlc are brilliant"
# → [2 3 4 5 6 0 0 0 0 0]
print("Example padded sequence:\n", padded_sequences[2])

Example padded sequence:
 [12 13  0  0  0  0  0  0  0  0  0  0  0  0  0]


**Step 4: Train-Test Split**

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.25, random_state=42               # test_size=0.25 → 25% data for testing, 75% for training.
                                                                            # random_state=42 → ensures same split every time (reproducibility)
)
# padded_sequences =
# [
#   [2, 3, 4, 0, 0],
#   [5, 6, 0, 0, 0],
#   [7, 8, 9, 0, 0],
#   [1, 2, 3, 4, 5]
# ]

# labels = [1, 0, 1, 0]
# After split (75% train, 25% test):

# X_train =
# [
#   [2, 3, 4, 0, 0],
#   [5, 6, 0, 0, 0],
#   [7, 8, 9, 0, 0]
# ]

# y_train = [1, 0, 1]

# X_test =
# [
#   [1, 2, 3, 4, 5]
# ]
# y_test = [0]


**Step 5: Build RNN Model**

In [5]:
embedding_dim = 50                               # Define embedding dimension (size of word vector representation)

rnn_model = Sequential([

    # Embedding Layer:
    # Converts word indices into dense vectors of fixed size (50 here)
    # input_dim = vocabulary size (max_words)
    # output_dim = embedding size (50)
    # input_length = length of each input sequence
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),

    # Simple RNN Layer:
    # 32 = number of hidden units (neurons)
    # Processes sequence step-by-step and captures temporal dependencies
    SimpleRNN(32),

    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Step 6: Train the Model**

In [6]:
rnn_history = rnn_model.fit(
    X_train, np.array(y_train),
    epochs=15,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)

Epoch 1/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 166ms/step - accuracy: 0.8333 - loss: 0.6571 - val_accuracy: 0.0000e+00 - val_loss: 0.9869
Epoch 2/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8333 - loss: 0.6226 - val_accuracy: 0.0000e+00 - val_loss: 1.0888
Epoch 3/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8333 - loss: 0.5210 - val_accuracy: 0.0000e+00 - val_loss: 1.2325
Epoch 4/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 0.3864 - val_accuracy: 0.0000e+00 - val_loss: 1.3471
Epoch 5/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8333 - loss: 0.4401 - val_accuracy: 0.0000e+00 - val_loss: 1.4505
Epoch 6/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 0.5016 - val_accuracy: 0.0000e+00 - val_loss: 1.5670
Epoch 7/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 0.4185 - val_accuracy: 0.0000e+00 - val_loss: 1.5708
Epoch 8/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8333 - loss: 0.4515 - val_accurac

**Step 7: Test a New Sentence**

In [7]:
new_text = ["I hate this movie but the ending is very bad"]

seq = tokenizer.texts_to_sequences(new_text)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
pred = rnn_model.predict(padded_seq)

print(f"Sentiment score: {pred[0][0]:.2f}")  # >0.5 → positive, <0.5 → negative

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
Sentiment score: 0.91


# **LSTM LONG SHORT TERM MEMORY**

**LSTM PIPELINE -**

**`Raw text`** → **`Tokenization & Padding`** → **`Train/Test Split`** → **`Embedding`** → **`LSTM`** → **`Dropout`** → **`Dense`** → **`Train`** → **`Evaluate`** → **`Predict Sentiment`**  


**Step 1: Import Libraries**

In [8]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

**Step 2: Prepare Dataset**

In [9]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative

**Step 3: Tokenize and Pad Sequences**

In [10]:
max_words = 1000
max_len = 15

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

print("Example padded sequence:\n", padded_sequences[6])

Example padded sequence:
 [ 2  6  3  4  7 20  5 21  0  0  0  0  0  0  0]


**Step 4: Train-Test Split**

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.25, random_state=42
)

**Step 5: Build LSTM Model**

In [12]:
embedding_dim = 50

lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # binary classification
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

 **Step 6: Train the Model**

In [13]:
lstm_history = lstm_model.fit(
    X_train, np.array(y_train),
    epochs=20,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 178ms/step - accuracy: 0.5000 - loss: 0.6869 - val_accuracy: 0.0000e+00 - val_loss: 0.7320
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8333 - loss: 0.6585 - val_accuracy: 0.0000e+00 - val_loss: 0.7721
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8333 - loss: 0.6499 - val_accuracy: 0.0000e+00 - val_loss: 0.8299
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8333 - loss: 0.6297 - val_accuracy: 0.0000e+00 - val_loss: 0.9007
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.5700 - val_accuracy: 0.0000e+00 - val_loss: 0.9869
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8333 - loss: 0.5360 - val_accuracy: 0.0000e+00 - val_loss: 1.0892
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8333 - loss: 0.5012 - val_accuracy: 0.0000e+00 - val_loss: 1.2709
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.4849 - val_accurac

**Step 7: Evaluate**

In [14]:
loss, accuracy = lstm_model.evaluate(X_test, np.array(y_test))
#print(f"Test Accuracy: {accuracy*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 347ms/step - accuracy: 0.0000e+00 - loss: 2.0236


**Step 8: Predict New Sentiment**

In [15]:
new_texts = [
    "I hate this movie but the ending is good",
    "Absolutely loved the story and characters"
]

seq = tokenizer.texts_to_sequences(new_texts)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
predictions = lstm_model.predict(padded_seq)

for text, pred in zip(new_texts, predictions):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    print(f"Text: {text}\nPredicted sentiment: {sentiment}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
Text: I hate this movie but the ending is good
Predicted sentiment: Positive

Text: Absolutely loved the story and characters
Predicted sentiment: Positive



# **GRU GATED RECURRENT UNIT**


**GRU PIPELINE -**

**`Raw text`** → **`Tokenization & Padding`** → **`Train/Test Split`** → **`Embedding`** → **`GRU`** → **`Dropout`** → **`Dense`** → **`Train`** → **`Evaluate`** → **`Predict Sentiment`**  

In [16]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from sklearn.model_selection import train_test_split

In [17]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative

In [18]:
max_words = 1000
max_len = 15

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.25, random_state=42
)

In [20]:
embedding_dim = 50

gru_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    GRU(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # binary classification
])

gru_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [21]:
gru_history = gru_model.fit(
    X_train, np.array(y_train),
    epochs=20,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 265ms/step - accuracy: 0.6667 - loss: 0.6970 - val_accuracy: 0.0000e+00 - val_loss: 0.7309
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.8333 - loss: 0.6525 - val_accuracy: 0.0000e+00 - val_loss: 0.7750
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.8333 - loss: 0.6435 - val_accuracy: 0.0000e+00 - val_loss: 0.8297
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.8333 - loss: 0.6061 - val_accuracy: 0.0000e+00 - val_loss: 0.8946
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.8333 - loss: 0.5759 - val_accuracy: 0.0000e+00 - val_loss: 0.9666
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8333 - loss: 0.5850 - val_accuracy: 0.0000e+00 - val_loss: 1.0457
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.5041 - val_accuracy: 0.0000e+00 - val_loss: 1.1563
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8333 - loss: 0.5079 - val_accurac

In [22]:
loss, accuracy = gru_model.evaluate(X_test, np.array(y_test))
#print(f"Test Accuracy: {accuracy*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step - accuracy: 0.0000e+00 - loss: 1.8174


In [23]:
new_texts = [
    "I hate this movie but the ending is good",
    "Absolutely loved the story and characters"
]

seq = tokenizer.texts_to_sequences(new_texts)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
predictions = gru_model.predict(padded_seq)

for text, pred in zip(new_texts, predictions):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    print(f"Text: {text}\nPredicted sentiment: {sentiment}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step
Text: I hate this movie but the ending is good
Predicted sentiment: Positive

Text: Absolutely loved the story and characters
Predicted sentiment: Positive



# **RNN - LSTM - GRU**

In [24]:
print("\n=== FINAL COMPARISON ===")

print(f"RNN  -> Train Acc: {rnn_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {rnn_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {rnn_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {rnn_history.history['val_loss'][-1]:.4f}")

print(f"LSTM -> Train Acc: {lstm_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {lstm_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {lstm_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {lstm_history.history['val_loss'][-1]:.4f}")

print(f"GRU  -> Train Acc: {gru_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {gru_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {gru_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {gru_history.history['val_loss'][-1]:.4f}")


=== FINAL COMPARISON ===
RNN  -> Train Acc: 1.0000, Loss: 0.1721, Val Acc: 0.0000, Val Loss: 2.0916
LSTM -> Train Acc: 0.8333, Loss: 0.3080, Val Acc: 0.0000, Val Loss: 2.0236
GRU  -> Train Acc: 0.8333, Loss: 0.4458, Val Acc: 0.0000, Val Loss: 1.8174


# **LSTM - IMDB DATASET**

**Step 1: Import Libraries**

In [25]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout ,Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

**Step 2: Load IMDB Dataset**

In [26]:
max_words = 20000  # vocabulary size

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_words)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Training samples: 25000
Testing samples: 25000


**Step 3: Pad Sequences**

In [27]:
max_len = 300

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

**Step 4: Build LSTM Model**

In [28]:
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(128),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Step 5: Train Model**

In [31]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 246s 783ms/step - accuracy: 0.9605 - loss: 0.1130 - val_accuracy: 0.8564 - val_loss: 0.4211
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 246s 785ms/step - accuracy: 0.9717 - loss: 0.0834 - val_accuracy: 0.8598 - val_loss: 0.4478
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 241s 772ms/step - accuracy: 0.9803 - loss: 0.0588 - val_accuracy: 0.8526 - val_loss: 0.5403
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 241s 770ms/step - accuracy: 0.9764 - loss: 0.0700 - val_accuracy: 0.8184 - val_loss: 0.5454
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 242s 773ms/step - accuracy: 0.9844 - loss: 0.0482 - val_accuracy: 0.8104 - val_loss: 0.7265


**Step 6: Evaluate**

In [32]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

782/782 ━━━━━━━━━━━━━━━━━━━━ 120s 154ms/step - accuracy: 0.7956 - loss: 0.7867
Test Accuracy: 79.56%
